installing dependencies


In [1]:
%pip install -r requirements.txt

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB 495.5 kB/s eta 0:00:26
   - -------------------------------------- 0.4/12.6 MB 3.3 MB/s eta 0:00:04
   ---- ----------------------------------- 1.5/12.6 MB 9.5 MB/s eta 0:00:02
   -------- ------------------------------- 2.7/12.6 MB 13.1 MB/s eta 0:00:01
   ------------- -------------------------- 4.2/12.6 MB 17.0 MB/s eta 0:00:01
   ----------------- ---------------------- 5.6/12.6 MB 18.9 MB/s eta 0:00:01
   -------------------- ------------------- 6.4/12.6 MB 19.4 MB/s eta 0:00:01
   ---------------------- ----------------- 7.2/12.6 MB 19.1 MB/s eta 0:00:01
   ----------------------- ---------------- 7.5/12.6 MB 17.1 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 16.7 MB/s eta 0:00:01
   --------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


loading the gemini api and creating the model

In [34]:
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
)

demo orders database


In [6]:
ORDERS_DB = {
     "12345":{
         "order_id":"12345",
         "customer_email":"john@example.com",
         "items":[
             {
                 "name":"Blue Cotton T-Shirt",
                 "price":29.99,
                 "quantity": 2
             },
             {
                 "name":"Denim Jeans",
                 "price":59.99,
                 "quantity": 1
             }
         ],
         "total":119.97,
         "order_date":"2025-01-15",
         "delivery_date": "2025-01-20",
         "status":"delivered",
         "tracking_number":"1z123543828"
     },
     "67890": {
         "order_id":"67890",
         "customer_email":"jain@example.com",
         "items":[
             {
               "name":"Red Kurta",
               "price":45.00,
               "quantity": 1  
             }
         ],
         "total":45.00,
         "order_date":"2025-01-20",
         "delivery_date":None,
         "status":"in_transit",
         "tracking_number":"1z123543856728"
     }
}

print(ORDERS_DB["12345"])

{'order_id': '12345', 'customer_email': 'john@example.com', 'items': [{'name': 'Blue Cotton T-Shirt', 'price': 29.99, 'quantity': 2}, {'name': 'Denim Jeans', 'price': 59.99, 'quantity': 1}], 'total': 119.97, 'order_date': '2025-01-15', 'delivery_date': '2025-01-20', 'status': 'delivered', 'tracking_number': '1z123543828'}


tool1: check the order status

In [7]:
from datetime import datetime

def check_order_status(order_id:str)->dict:
    """looks up order by its order ID"""

    order = ORDERS_DB.get(order_id) # if no order is present it will return None
    if order is None:
        return {
            "error": f"order {order_id} is not found",
            "source":"orders_api"
        }
    return {
        "data": order,
        "source": "orders_api",
        "retrieved_at": datetime.now().isoformat()
    }


tool2: refund calculator

In [8]:
def calculate_refund(order_id:str,reason:str) -> dict:
    """calculate the potential refund using demo return policy"""

    order = ORDERS_DB.get(order_id)
    if order is None:
        return {
            "error": f"order {order_id} is not found",
            "source":"refund_calculator"
        }

    reason = reason.lower().strip()

    if reason in {"damaged","defective"}:
        fee = 0.0
    elif reason == "change_of_mind":
        fee = order["total"]*0.15
    else:
        return {
           "error":"Unsupported refund reason",
           "source":"refund_caculator"
        }
    return {
        "data":{
            "order_id": order_id,
            "original_amount": order["total"],
            "restocking_fee": round(fee,2),
            "refund_amount": round(
                order["total"]-fee,2
            ),
            "reason":reason
        },
        "source":"refund_calculator"
    }
        

tool3: process the return


In [9]:
def process_return(order_id:str,reason:str)->dict:
    """check weather a return be initiated"""

    order = ORDERS_DB.get(order_id)

    if order is None:
        return {
            "error": f"order {order_id} is not found",
            "source":"returns_api"
        }

    if order["status"] != "delivered":
        return {
            "error":"order has not been delivered",
            "source":"returns_api"
        }

    delivery_date = datetime.strptime(order["delivery_date"],"%Y-%m-%d").date()

    days_since_delivery = (datetime.now().date()-delivery_date).days

    if days_since_delivery > 30:
        return {
            "error":"return window is expired (30 days)",
            "source":"returns_api",
            "policy_reference":"return_policy.md#eligibility"
        }
    return {
        "data":{
            "order_id":order_id,
            "return_eligible":True,
            "reason":reason,
            "message":"eligible for return"
        },
        "source":"returns_api"
    }

    

tool5: return policy

In [10]:
POLICIES = [
    {
        "policy":"return_policy",
        "text":(
            "items can be returned within 30 days of delivery."
            "damaged or defective items can be returned for full refund."
            "change of mind returns are subject to 15% resticking fee."
        )
    }
]

def retrieve_policy(query:str)->dict:
    """retrieves the policy based on the query."""
    return {
        "data": {
            "query":query,
            "policy":POLICIES
        },
        "source":"knowledge_base"
    }

convert the functions into langchain tools 

In [11]:
from langchain_core.tools import StructuredTool
tools = [
    StructuredTool.from_function(check_order_status),
    StructuredTool.from_function(calculate_refund),
    StructuredTool.from_function(process_return),
    StructuredTool.from_function(retrieve_policy),
]

for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"description: {tool.description}")
    print()

Tool: check_order_status
description: looks up order by its order ID

Tool: calculate_refund
description: calculate the potential refund using demo return policy

Tool: process_return
description: check weather a return be initiated

Tool: retrieve_policy
description: retrieves the policy based on the query.



creating an agent state

In [12]:
from typing import Any,NotRequired
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    """The state of the agent"""
    query: str
    messages: list[BaseMessage]
    tools_results: list[dict[str,Any]]
    sources: list[str]
    steps: int #3
    max_steps: int #10
    plan: NotRequired[dict[str,Any]]
    final_response: str

build planner

In [13]:
#plan
#action
#tools
#tools arg
#reason for choosing the tool

from pydantic import BaseModel , Field
from typing import Literal 

class Plan(BaseModel):
    action: Literal["tool","finish"] = Field(   #either u need to use the tool or finish the task without using tool
        description = "call a tool or finish the task"
    )
 
    tool_name: str | None = Field(
        default= None,
        description="the name of the tool to call, if the action is 'tool'. "
    )

    tool_args: dict[str,Any] = Field(
        default_factory = dict,
        description = "the arguements to pass to the tool, if action is 'tool'."
    )


    reasoning_summary:str = Field(
        description= "A summary of the reasoning behind the plan."
    )

In [14]:
planner_llm = llm.with_structured_output(Plan)

In [15]:
TOOL_NAMES = {tool.name for tool in tools}
def planner(state:AgentState) -> dict:
    prompt = f"""
     You are an e-commerce customer support agent.

     User question: {state['query']}
     
     Previous tools results: {state['tools_results']}

     Available_tools : {','.join(TOOL_NAMES)}
     
     Choose one of the following actions:
     1.Call a relevant tool to get more information ony if needed.
     2.Finish only you have enough information to answer the user's question.
     3.Never invent information. if you don't know the answer say so.
     4.Never claim that you have successfully called a tool if you have'nt . Only use the tools provided.
"""
    plan = planner_llm.invoke(prompt)
    if plan.action == "tool" and plan.tool_name not in TOOL_NAMES:
        return {
            "final_response": f"Error: Tool '{plan.tool_name}' is not available.",
            "steps":state["max_steps"],
        }
    return {
        "message":state["messages"],
        "steps":state["steps"]+1,
        "tools_results":state["tools_results"],
        "sources":state["sources"],
        "final_response":(
            plan.model_dump_json()
        ),
    }


tool executor

In [16]:
TOOL_MAP = {tool.name: tool for tool in tools} #list of tools

def execute_tool(tool_name: str,tool_args: dict) -> dict:
    """Execute a tool by name with the provided arguemnets."""

    tool = TOOL_MAP.get(tool_name)

    if tool is None:
        return {
            "error":f"Tool '{tool_name}' not found.",
            "source":"tool_executor"
        }
    
    try:
        result = tool.invoke(tool_args)
        return{
            "data":result,
            "tool":tool_name
        }
    except Exception as e:
        return {
            "error":str(e),
            "source":tool_name
        }


final responder

In [17]:
from langchain_core.messages import HumanMessage , SystemMessage

def generate_final_answer(query:str,tools_results:list[dict])->str:
    """generate the final answer based on the user query and tool results."""

    evidence = []
    sources = []

    for result in tools_results:
        source = result.get("source","unknown")
        evidence.append({
            "source":source,
            "result":result
        })
        
        if source not in sources:
            sources.append(source)

    response = llm.invoke([SystemMessage(content = {
         "Answer the users questions based on the provided evidence"
         "Do not invent facts. if the evidence is insufficient, say so."
         "Refund from an approoved or issued refund."
         "Cite facts from the evidence and provide sources in the answer."
         "If the evidence is insufficient or missing , say so."
    }),
    HumanMessage(content={
         f"User question: {query}\n"
         f"Evidence: {evidence}\n"
         f"Sources: {sources}"

    }),])

    return response.content



connect everything with langgraph

Planner node

In [18]:
def plan_step(state: AgentState)->dict:

    plan = planner_llm.invoke(f"""
You are an e-commerce support agent.

User question: {state['query']}

Previous tools results: {state['tools_results']}

Available tools: {', '.join(TOOL_NAMES)}

Select one of the tools, or finish if sufficient information and evidence is available.Never invent information. if you don't know, say so.only use the tools provided.
"""
)
    return {
        "plan":plan.model_dump(),
        "steps":state["steps"]+1
    }

tool node

In [19]:
def run_tool(state:AgentState)->dict:
    plan = state["plan"]
    
    result = execute_tool(plan["tool_name"],plan["tool_args"])
    
    results = state["tools_results"]+ [result]
    sources = list(state["sources"])
    
    source = result.get("source","unknown")
    if source not in sources:
        sources.append(source)
    
    return {
        "tools_results":results,
        "sources":sources
    }


In [20]:
def respond(state:AgentState)->dict:

    answer = generate_final_answer(state["query"],state["tools_results"])
    return {
        "final_response":answer
    }
    

defining routing logic

In [21]:
from langgraph.graph import StateGraph, START, END

def route_after_planning(state: AgentState) -> str:
    plan = state.get("plan", {})

    if state["steps"] >= state["max_steps"]:
        return "respond"

    if plan.get("action") == "finish":
        return "respond"

    if (
        plan.get("action") == "tool"
        and plan.get("tool_name") in TOOL_NAMES
    ):
        return "run_tool"

    return "respond"

In [22]:
def build_agent():
    builder = StateGraph(AgentState)

    builder.add_node("plan_step", plan_step)
    builder.add_node("run_tool", run_tool)
    builder.add_node("respond", respond)

    builder.add_edge(START, "plan_step")

    builder.add_conditional_edges(
        "plan_step",
        route_after_planning,
        {
            "run_tool": "run_tool",
            "respond": "respond"
        }
    )

    builder.add_edge("run_tool", "plan_step")
    builder.add_edge("respond", END)

    return builder.compile()

In [30]:
def ask_agent(query: str) -> str:
    result = build_agent().invoke({
        "query": query,
        "tools_results": [],
        "messages": [],
        "sources": [],
        "steps": 0,
        "max_steps": 10
    })

    return result["final_response"]

In [31]:
from pprint import pprint

In [35]:
pprint(ask_agent("where is my order #12345"))

[{'extras': {'signature': 'EoINCv8MAWkUfRN7bBuKb3SefioBE3qD0NvXMKb86+GGa1XpBXMTLy2c1iWDtWFE3F2W1fQ7rDSkB6U4bWG4vYn9YGEo6T3v4ravt+adx7YIp7wOMl2TLbvJeK2/EBDd0C1YkRn+P11yKeEkMg82KwGD00dleJ9LnyQuuu9f9Ggeq+kHwUnRzOKYkaEw61gK9MrmgEGfrqrJgBuVmGHrlftfoIPqKjH0Gmc5MCD0aJDDGt6shmgdQkYuHufLhUt7xwB5JsETixDNpNuHsSc/+lmm9ETz4N2Dn/JC+cRvCglnVoPNzag9pDrni7qKQTh0RnCFfuFRfAN6iO84wRhGXfkjesPaHwdBjoqMQs2Cdh1SRcDv7G3ZWBseE+/9UEWjLb0uV/wT0RGYSnyf72OvxqpwyKkZaUPxXUR4LLDqYhNK/PG+He1JXjEeV/I9+Kcd5Cb+N2HPQB/i8SR1+8rPwTc6sjGUtfn49IVr3OVf6iQhJjuyqh3oVQVoxLWKnF6ei0Kh6KQbkls7kQyYeiBHMxmvDkr8HAm0lO9BWTCEYe94NxIBb+bJgnM3zr4shDZ9yw3pGr86oVd6qmcN2bpXSIYFcdq5zhgfXkGIiAVcD55m+gs1pZ0JMM95XK091UVzi46MJf52ABQjPqz9JH2VdbWuSxp0glZMibPXHvGL9j88sHm0rBqhgF6kNviSpKNvD3S3tuzAYiYMYJ+o64+4zszNV1VmsEdueaqR4fQojW4KKdkwvRgZoyNPKdcKg9Ogl/OYywLKtmyyGrQ++WS+qHp4h8Vo9onazvRCa4q6V0lQti6ZgFwYEe6VFIATjGZ1Bnv7y/f43nnD5myE3L2aON2Nh38oJHlxs+RdBzyTTmxqUt7dQKxP/pl1K+vzzD9Dcy6QbqyLtFhpIEcavFlmDxoLhIkxKbn7GD84H4X0YO6RJV1OzrUCIr46/+Sj6gm9+fAtNhcYhkqq1